In [2]:
import os
import re
import random
from tqdm import tqdm

random.seed(42)
CHARACTER_MAP = "gkamztlbdqiyfucxbhsjoprnweygtjmevchdxsanqolkrvwiypjzquhe"

def encode_line(text: str) -> str:
    return "".join(CHARACTER_MAP[ord(c) % 56] if c not in "\n\t\r" else c for c in text)

RAW_WIKI = "corpus/raw_wiki.txt"
RAW_EF = "corpus/raw_ef.txt"
OUT_DIR = "corpus"
os.makedirs(OUT_DIR, exist_ok=True)

## Load & Normalize Raw Lines

In [3]:
def load_normalized_lines(path):
    if not os.path.exists(path):
        return []
    with open(path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]
    
    valid = []
    for line in lines:
        clean = re.sub(r"\s+", " ", line)
        if 20 <= len(clean) <= 200:
            if not re.search(r"[。！？；]$", clean):
                clean += "。"
            valid.append(clean)
    return valid

print("Loading normalized raw data...")
wiki_lines = load_normalized_lines(RAW_WIKI)
ef_lines = load_normalized_lines(RAW_EF)
print(f"Wikipedia: {len(wiki_lines)} | Endfield: {len(ef_lines)}")

Loading normalized raw data...
Wikipedia: 17879714 | Endfield: 13096


## Save Dedicated Endfield Files (for tokenizer prioritization)

In [4]:
EF_ZH_PATH = os.path.join(OUT_DIR, "endfield.zh")
EF_SKZ_PATH = os.path.join(OUT_DIR, "endfield.skz")

with open(EF_ZH_PATH, "w", encoding="utf-8") as f_zh, \
     open(EF_SKZ_PATH, "w", encoding="utf-8") as f_skz:
    for zh in ef_lines:
        f_zh.write(zh + "\n")
        f_skz.write(encode_line(zh) + "\n")

print(f"Saved dedicated Endfield corpus: {len(ef_lines)} pairs")

Saved dedicated Endfield corpus: 13096 pairs


## Merge Corpus

In [5]:
def load_txt(path):
    if not os.path.exists(path): return []
    with open(path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]

wiki_lines = load_txt(RAW_WIKI)
ef_lines = load_txt(RAW_EF)
merged = wiki_lines + ef_lines
print(f"Merged corpus: Wiki {len(wiki_lines)} + EF {len(ef_lines)} = Total {len(merged)} lines")

Merged corpus: Wiki 17880809 + EF 13096 = Total 17893905 lines


## Shuffle & Split

In [6]:
random.shuffle(merged)
split_idx = int(len(merged) * 0.95)
train_data = merged[:split_idx]
val_data = merged[split_idx:]
print(f"Split complete: Train {len(train_data)} | Val {len(val_data)}")

Split complete: Train 16999209 | Val 894696


## Atomic Parellel Write

In [7]:
def write_parallel(data, prefix):
    zh_path = os.path.join(OUT_DIR, f"{prefix}.zh")
    skz_path = os.path.join(OUT_DIR, f"{prefix}.skz")
    with open(zh_path, 'w', encoding='utf-8') as f_zh, \
         open(skz_path, 'w', encoding='utf-8') as f_skz:
        for line in tqdm(data, desc=f"Writing {prefix}"):
            f_zh.write(line + '\n')
            f_skz.write(encode_line(line) + '\n')

write_parallel(train_data, "train")
write_parallel(val_data, "val")

Writing val: 100%|██████████| 894696/894696 [00:09<00:00, 98929.91it/s] 


## Consistency Check

In [8]:
train_zh_lines = len(open("corpus/train.zh", encoding='utf-8').readlines())
train_skz_lines = len(open("corpus/train.skz", encoding='utf-8').readlines())
val_zh_lines = len(open("corpus/val.zh", encoding='utf-8').readlines())
val_skz_lines = len(open("corpus/val.skz", encoding='utf-8').readlines())

assert train_zh_lines == train_skz_lines, f"Train misalignment! zh:{train_zh_lines} vs skz:{train_skz_lines}"
assert val_zh_lines == val_skz_lines, f"Val misalignment! zh:{val_zh_lines} vs skz:{val_skz_lines}"

print("Stage One complete: corpus unified, strictly aligned, validation set split.")
print(f"Output: {OUT_DIR}/{{train,val}}.{{zh,skz}}")

Stage One complete: corpus unified, strictly aligned, validation set split.
Output: corpus/{train,val}.{zh,skz}
